<a href="https://colab.research.google.com/github/shivangi221b/ASR-Fine-Tuning-Modules-and-RL-Adaptation-Analysis/blob/shivangi_initial_setup/STT_Domain_Adaptation_HF_Transformers_Practical_Validation_Script_(LibriSpeech).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STT Domain Adaptation - Practical Validation Script
Goal:
  1. Baseline fine-tuning on ~Common Voice 17.0~ LibriSpeech(1000 samples)
  2. Document training loop behaviour + hook points
  3. Test all three custom loss/reward injections:
       - MWER  (WER-based policy gradient)
       - WWER  (Weighted WER - upweights domain terms)
       - LLM   (Mock LLM scorer placeholder)

Model  : openai/whisper-base  (fast on A100, \~74M params)
Runtime: Colab Pro A100 recommended (\~30 min full run)
         Free T4 also works but limit TRAIN_SAMPLES to 200

To run on GCP later: see GCP_INSTRUCTIONS.md (generated alongside this file)

# CELL 1 — Install dependencies

In [ ]:
!pip install -q \
    transformers==4.40.0 \
    datasets==2.19.0 \
    evaluate==0.4.1 \
    jiwer==3.0.3 \
    accelerate==0.29.3 \
    soundfile \
    librosa \
    torch>=2.0

# !pip install -q \
#     transformers>=4.41.0 \
#     datasets==2.19.0 \
#     evaluate==0.4.1 \
#     jiwer==3.0.3 \
#     accelerate==0.29.3 \
#     soundfile \
#     librosa \
#     fsspec==2025.3.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [ ]:
!pip install -q "accelerate>=0.34.0" "transformers>=4.41.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.6 MB/s eta 0:00:00


# CELL 2 — Imports & Config

In [1]:
import os
import time
import json
import logging
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import torch
import numpy as np
from torch import nn
from torch.utils.data import DataLoader

from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)
import evaluate

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [22]:
# ---- CONFIG (edit these) ----
MODEL_NAME       = "openai/whisper-base"       # processor always loads from here
SFT_CHECKPOINT   = "/content/drive/MyDrive/ResearchUnderProfChen/whisper-cv17-baseline-latest"   # model weights load from here
LANGUAGE         = "en"
TASK             = "transcribe"
TRAIN_SAMPLES    = 200   # reduce to 200 for T4 free tier, OG is 1000
EVAL_SAMPLES     = 200
BATCH_SIZE       = 16     # A100: 16 fine; T4: use 8, OG is 16
LEARNING_RATE    = 1e-6
NUM_EPOCHS       = 5      # 3 epochs ~25 min on A100
OUTPUT_DIR       = "./whisper-cv17-baseline"
REWARD_MODE      = "mwer"  # "mwer" | "wwer" | "llm" | "all"

# Domain-specific terms for WWER (simulate healthcare/judicial vocab)
DOMAIN_TERMS = {
    # healthcare examples
    "myocardial", "infarction", "hypertension", "tachycardia",
    "arrhythmia", "stethoscope", "auscultation", "echocardiogram",
    # judicial examples
    "plaintiff", "defendant", "affidavit", "subpoena",
    "jurisdiction", "deposition", "mandamus", "injunction",
}
DOMAIN_TERM_WEIGHT = 3.0   # domain term errors count 3x in WWER

# CELL 3 — Load Dataset (Common Voice 17.0, English subset)


In [3]:
def load_dataset_librispeech(train_n: int, eval_n: int):
    from datasets import load_dataset
    logger.info("Loading LibriSpeech clean-100 ...")
    ds = load_dataset(
        "librispeech_asr",
        "clean",
        split={
            "train":      f"train.100[:{train_n}]",
            "validation": f"validation[:{eval_n}]",
        },
        trust_remote_code=True,
    )
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    logger.info(f"  Train: {len(ds['train'])} | Eval: {len(ds['validation'])}")
    return ds

# CELL 4 — Processor & Feature Extraction


In [4]:
def build_processor():
    processor = WhisperProcessor.from_pretrained(
        MODEL_NAME, language=LANGUAGE, task=TASK
    )
    return processor

In [5]:
def prepare_dataset(batch, processor):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    ).input_features[0]
    # handle both CV17 ("sentence") and LibriSpeech ("text")
    text = batch.get("sentence") or batch.get("text", "")
    batch["labels"] = processor.tokenizer(text).input_ids
    return batch

# CELL 5 — Data Collator


In [24]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # Pad input features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        # Pad labels, replace pad token with -100 so loss ignores padding
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# CELL 6 — Evaluation Metric


In [7]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred, processor):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

 # CELL 7 — HOOK POINTS (where RL reward injection happens)
 These are the three "hook points" documented for the paper.
In a real RL training loop you would:
   1. Forward pass → get logits
   2. Sample hypotheses from logits (not just argmax)
   3. Score each hypothesis with one of the reward functions below
   4. Weight the cross-entropy loss by (reward - baseline)
   5. Backward pass on weighted loss

In [8]:
# ---- REWARD FUNCTION 1: MWER (Minimum Word Error Rate) ----

def compute_mwer_reward(hypotheses: List[str], references: List[str]) -> torch.Tensor:
  """
  MWER reward: reward = 1 - WER (higher = better transcription)
  Used in policy gradient: loss = -log_prob * (reward - baseline)

  Hook point: after beam search / sampling, before loss.backward()

  Args:
      hypotheses : list of decoded hypothesis strings (batch)
      references : list of ground truth strings (batch)
  Returns:
      rewards    : tensor of shape (batch,) in range [0, 1]
  """
  from jiwer import wer as compute_wer
  rewards = []
  for hyp, ref in zip(hypotheses, references):
      if len(ref.strip()) == 0:
          rewards.append(0.5)   # neutral reward for empty reference
          continue
      error_rate = compute_wer(ref, hyp)
      reward = max(0.0, 1.0 - error_rate)   # clip to [0,1]
      rewards.append(reward)
  return torch.tensor(rewards, dtype=torch.float32)

In [9]:
# ---- REWARD FUNCTION 2: WWER (Weighted WER - domain term boost) ----

def compute_wwer_reward(
    hypotheses: List[str],
    references: List[str],
    domain_terms: set = DOMAIN_TERMS,
    term_weight: float = DOMAIN_TERM_WEIGHT,
) -> torch.Tensor:
    """
    WWER reward: like MWER but domain term errors penalised more heavily.
    Domain terms in DOMAIN_TERMS get weight=term_weight (default 3x).
    All other words get weight=1.0.

    Hook point: same as MWER — after decoding, before loss.backward()

    Args:
        hypotheses   : list of hypothesis strings
        references   : list of reference strings
        domain_terms : set of domain-specific terms (lowercase)
        term_weight  : how much more to penalise domain term errors
    Returns:
        rewards      : tensor of shape (batch,) in range [0, 1]
    """
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        ref_words = ref.lower().split()
        hyp_words = hyp.lower().split()

        if not ref_words:
            rewards.append(0.5)
            continue

        # Build per-word weights for reference
        weights = [
            term_weight if w in domain_terms else 1.0
            for w in ref_words
        ]
        total_weight = sum(weights)

        # Simple weighted substitution count (positional alignment)
        # For production: use proper edit-distance with weights
        weighted_errors = 0.0
        min_len = min(len(ref_words), len(hyp_words))
        for i in range(min_len):
            if ref_words[i] != hyp_words[i]:
                weighted_errors += weights[i] if i < len(weights) else 1.0
        # Penalise length difference
        length_diff = abs(len(ref_words) - len(hyp_words))
        weighted_errors += length_diff * 1.0

        wwer = weighted_errors / total_weight
        reward = max(0.0, 1.0 - wwer)
        rewards.append(reward)

    return torch.tensor(rewards, dtype=torch.float32)

In [10]:
# ---- REWARD FUNCTION 3: LLM Scorer (mock placeholder) ----

def compute_llm_reward(
    hypotheses: List[str],
    references: List[str],
    domain: str = "healthcare",  # or "judicial"
    use_mock: bool = True,       # set False when real LLM is connected
) -> torch.Tensor:
    """
    LLM-based reward: a domain-aware LLM evaluates transcription quality.
    Returns a score in [0, 1].

    In production:
      - Send (hypothesis, reference, domain_context) to an LLM
      - LLM returns a JSON score {"score": 0.85, "reason": "..."}
      - Use score as reward signal

    Hook point: same as MWER/WWER — after decoding, before loss.backward()
    NOTE: LLM calls add latency (~200-500ms per batch).
          In real training: batch calls, cache identical refs, use smaller LLM.

    Args:
        hypotheses : decoded hypothesis strings
        references : ground truth strings
        domain     : "healthcare" or "judicial" (affects LLM prompt)
        use_mock   : if True, returns a simulated score (for testing)
    Returns:
        rewards    : tensor of shape (batch,) in range [0, 1]
    """
    if use_mock:
        return _mock_llm_reward(hypotheses, references)
    else:
        return _real_llm_reward(hypotheses, references, domain)

In [11]:
def _mock_llm_reward(hypotheses, references):
    """
    Mock: simulates LLM scoring by blending WER score + random noise.
    Useful for testing the training loop without API costs.
    """
    from jiwer import wer as compute_wer
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        if not ref.strip():
            rewards.append(0.5)
            continue
        base_score = max(0.0, 1.0 - compute_wer(ref, hyp))
        noise = np.random.normal(0, 0.05)              # small noise to simulate LLM variance
        score = float(np.clip(base_score + noise, 0, 1))
        rewards.append(score)
    return torch.tensor(rewards, dtype=torch.float32)

In [12]:
def _real_llm_reward(hypotheses, references, domain):
    """
    Real LLM reward — wire this up to your LLM API of choice.
    Shown here with a Claude API call structure.
    Replace with any LLM (GPT-4, Llama, domain fine-tuned model).
    """
    import anthropic  # pip install anthropic

    client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var

    SYSTEM_PROMPT = f"""You are a {domain} transcription quality evaluator.
    Given a hypothesis transcription and a reference transcription,
    score the hypothesis from 0.0 to 1.0 where:
      1.0 = perfect transcription, all {domain} terms correct
      0.5 = partially correct, some errors
      0.0 = completely wrong

    Pay special attention to {domain}-specific terminology.
    Respond ONLY with a JSON object: {{"score": <float>}}"""

    rewards = []
    for hyp, ref in zip(hypotheses, references):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",  # fast + cheap for reward scoring
                max_tokens=50,
                system=SYSTEM_PROMPT,
                messages=[{
                    "role": "user",
                    "content": f"Reference: {ref}\nHypothesis: {hyp}"
                }]
            )
            result = json.loads(response.content[0].text)
            rewards.append(float(result["score"]))
        except Exception as e:
            logger.warning(f"LLM reward call failed: {e}. Using fallback 0.5")
            rewards.append(0.5)

    return torch.tensor(rewards, dtype=torch.float32)

# CELL 8 — Custom Trainer with Reward Loss Injection

In [26]:
class RewardAwareTrainer(Seq2SeqTrainer):
    """
    Extends HuggingFace Seq2SeqTrainer to inject reward-based loss.

    The compute_loss method is the primary hook point.
    Standard training uses cross-entropy loss only.
    With reward injection: loss = CE_loss * (1 - reward) + RL_loss * reward_weight

    This is a simplified MWER-style injection. For full policy gradient:
      - You'd sample N hypotheses per input (not just argmax)
      - Compute per-hypothesis rewards
      - Use REINFORCE-style gradient update
    """

    def __init__(self, *args, processor=None, reward_mode="mwer", reward_weight=0.3, **kwargs):
        super().__init__(*args, **kwargs)
        self.processor    = processor
        self.reward_mode  = reward_mode
        self.reward_weight = reward_weight
        self._step_log   = []   # for logging hook point behaviour

    # def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
    #     """
    #     HOOK POINT: This is where reward injection happens.
    #     Currently: CE loss only (baseline behaviour).
    #     To add RL: uncomment the reward block below.
    #     """
    #     labels = inputs.get("labels")

    #     # --- Standard forward pass ---
    #     outputs = model(**inputs)
    #     ce_loss = outputs.loss

    #     # --- REWARD INJECTION BLOCK (currently for demonstration) ---
    #     # In full RL training you would:
    #     #   1. Generate hypotheses: model.generate(inputs["input_features"])
    #     #   2. Decode: processor.batch_decode(hypothesis_ids)
    #     #   3. Decode references from labels
    #     #   4. Score with reward function
    #     #   5. Combine with CE loss

    #     # Uncomment below to activate reward injection:
    #     # ------------------------------------------------
    #     # with torch.no_grad():
    #     #     hyp_ids  = model.generate(inputs["input_features"])
    #     #     ref_ids  = labels.clone()
    #     #     ref_ids[ref_ids == -100] = self.processor.tokenizer.pad_token_id
    #     #     hyps = self.processor.tokenizer.batch_decode(hyp_ids, skip_special_tokens=True)
    #     #     refs = self.processor.tokenizer.batch_decode(ref_ids, skip_special_tokens=True)

    #     # if self.reward_mode == "mwer":
    #     #     rewards = compute_mwer_reward(hyps, refs).to(ce_loss.device)
    #     # elif self.reward_mode == "wwer":
    #     #     rewards = compute_wwer_reward(hyps, refs).to(ce_loss.device)
    #     # elif self.reward_mode == "llm":
    #     #     rewards = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)

    #     # penalty = 1.0 - rewards.mean()   # high reward = low penalty
    #     # total_loss = ce_loss + self.reward_weight * penalty
    #     # ------------------------------------------------

    #     with torch.no_grad():
    #         hyp_ids = model.generate(inputs["input_features"])
    #         ref_ids  = inputs["labels"].clone()
    #         ref_ids[ref_ids == -100] = self.processor.tokenizer.pad_token_id
    #         hyps = self.processor.tokenizer.batch_decode(hyp_ids, skip_special_tokens=True)
    #         refs = self.processor.tokenizer.batch_decode(ref_ids, skip_special_tokens=True)

    #     if self.reward_mode == "mwer":
    #         rewards = compute_mwer_reward(hyps, refs).to(ce_loss.device)
    #     elif self.reward_mode == "wwer":
    #         rewards = compute_wwer_reward(hyps, refs).to(ce_loss.device)
    #     elif self.reward_mode == "llm":
    #         rewards = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)
    #     elif self.reward_mode == "all":
    #         # Average all three reward signals together
    #         mwer = compute_mwer_reward(hyps, refs).to(ce_loss.device)
    #         wwer = compute_wwer_reward(hyps, refs).to(ce_loss.device)
    #         llm  = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)
    #         rewards = (mwer + wwer + llm) / 3.0
    #     else:
    #         # Fallback: no reward signal, just use CE loss unmodified
    #         logger.warning(f"Unknown reward_mode '{self.reward_mode}', skipping reward injection")
    #         rewards = torch.ones(ce_loss.shape[0] if ce_loss.dim() > 0 else 1).to(ce_loss.device)

    #     penalty    = 1.0 - rewards.mean()
    #     total_loss = ce_loss + self.reward_weight * penalty

    #     # For now: just log that we passed through the hook
    #     self._step_log.append({
    #         "step": self.state.global_step,
    #         "ce_loss": ce_loss.item(),
    #         "reward_mode": self.reward_mode,
    #     })

    #     return (ce_loss, outputs) if return_outputs else ce_loss

    # def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
    #     labels = inputs.get("labels")
    #     outputs = model(**inputs)
    #     ce_loss = outputs.loss

    #     USE_REWARD_EVERY_N_STEPS = 4

    #     with torch.no_grad():
    #       if self.state.global_step % USE_REWARD_EVERY_N_STEPS == 0:
    #           # FIX: use greedy decode with strict length cap instead of beam search
    #           # generate() with default settings can deadlock on long/silent samples
    #           hyp_ids = model.generate(
    #               inputs["input_features"],
    #               max_new_tokens=100,       # hard cap — prevents infinite loops
    #               num_beams=1,              # greedy, not beam search — much faster + no deadlock
    #               do_sample=False,
    #               # max_length=None,   # ADD THIS — clears the conflicting config value
    #           )
    #           ref_ids = inputs["labels"].clone()
    #           ref_ids[ref_ids == -100] = self.processor.tokenizer.pad_token_id
    #           hyps = self.processor.tokenizer.batch_decode(hyp_ids, skip_special_tokens=True)
    #           refs = self.processor.tokenizer.batch_decode(ref_ids, skip_special_tokens=True)
    #           self._last_hyps = hyps
    #           self._last_refs = refs
    #       else:
    #           hyps = getattr(self, "_last_hyps", None)
    #           refs = getattr(self, "_last_refs", None)

    #     # guard: skip reward on first few steps before first generate
    #     if hyps is None or refs is None:
    #         return (ce_loss, outputs) if return_outputs else ce_loss

    #     if self.reward_mode == "mwer":
    #         rewards = compute_mwer_reward(hyps, refs).to(ce_loss.device)
    #     elif self.reward_mode == "wwer":
    #         rewards = compute_wwer_reward(hyps, refs).to(ce_loss.device)
    #     elif self.reward_mode == "llm":
    #         rewards = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)
    #     elif self.reward_mode == "all":
    #         mwer = compute_mwer_reward(hyps, refs).to(ce_loss.device)
    #         wwer = compute_wwer_reward(hyps, refs).to(ce_loss.device)
    #         llm  = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)
    #         rewards = (mwer + wwer + llm) / 3.0
    #     else:
    #         logger.warning(f"Unknown reward_mode '{self.reward_mode}', skipping reward injection")
    #         rewards = torch.ones(1).to(ce_loss.device)

    #     penalty    = 1.0 - rewards.mean()
    #     total_loss = ce_loss + self.reward_weight * penalty

    #     self._step_log.append({
    #         "step": self.state.global_step,
    #         "ce_loss": ce_loss.item(),
    #         "total_loss": total_loss.item(),
    #         "reward_mode": self.reward_mode,
    #     })

    #     # return (ce_loss, outputs) if return_outputs else ce_loss
    #     return (total_loss, outputs) if return_outputs else total_loss


    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        USE_REWARD_EVERY_N_STEPS = 4

        with torch.no_grad():
            if self.state.global_step % USE_REWARD_EVERY_N_STEPS == 0:
                # hyp_ids = model.generate(
                #     inputs["input_features"],
                #     max_new_tokens=100,
                #     num_beams=1,
                #     do_sample=False,
                # )

                hyp_ids = model.generate(
                    inputs["input_features"],
                    max_new_tokens=50,   # reduced from 100 — SFT model is fluent, doesn't need 100
                    num_beams=1,
                    do_sample=False,
                    language="en",       # ADD: explicit language forces shorter paths
                    task="transcribe",   # ADD: explicit task
                )
                ref_ids = inputs["labels"].clone()
                ref_ids[ref_ids == -100] = self.processor.tokenizer.pad_token_id
                hyps = self.processor.tokenizer.batch_decode(hyp_ids, skip_special_tokens=True)
                refs = self.processor.tokenizer.batch_decode(ref_ids, skip_special_tokens=True)
                self._last_hyps = hyps
                self._last_refs = refs
            else:
                hyps = getattr(self, "_last_hyps", None)
                refs = getattr(self, "_last_refs", None)

        if hyps is None or refs is None:
            return (ce_loss, outputs) if return_outputs else ce_loss

        if self.reward_mode == "mwer":
            rewards = compute_mwer_reward(hyps, refs).to(ce_loss.device)
        elif self.reward_mode == "wwer":
            rewards = compute_wwer_reward(hyps, refs).to(ce_loss.device)
        elif self.reward_mode == "llm":
            rewards = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)
        elif self.reward_mode == "all":
            mwer = compute_mwer_reward(hyps, refs).to(ce_loss.device)
            wwer = compute_wwer_reward(hyps, refs).to(ce_loss.device)
            llm  = compute_llm_reward(hyps, refs, use_mock=True).to(ce_loss.device)
            rewards = (mwer + wwer + llm) / 3.0
        else:
            rewards = torch.ones(1).to(ce_loss.device)

        penalty    = 1.0 - rewards.mean()
        total_loss = ce_loss + self.reward_weight * penalty

        self._step_log.append({
            "step":        self.state.global_step,
            "ce_loss":     ce_loss.item(),
            "total_loss":  total_loss.item(),
            "reward_mean": rewards.mean().item(),
            "reward_mode": self.reward_mode,
        })

        return (total_loss, outputs) if return_outputs else total_loss

# CELL 9 — Training Loop Logger (Callback)

In [27]:
class TrainingLogger(TrainerCallback):
    def __init__(self):
        self.epoch_start_time = None
        self.step_logs = []

    def on_train_begin(self, args, state, control, **kwargs):
        # FIX: explicitly reset state at train start so re-entrant eval calls
        # triggered by load_best_model_at_end don't crash with uninitialised state
        self.epoch_start_time = time.time()

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()
        logger.info(f"\n{'='*50}")
        logger.info(f"EPOCH {int(state.epoch) + 1} STARTED")
        logger.info(f"{'='*50}")

    def on_epoch_end(self, args, state, control, **kwargs):
        # FIX: guard against None if epoch_begin was never called
        elapsed = (time.time() - self.epoch_start_time) if self.epoch_start_time else 0
        mem_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        logger.info(f"EPOCH {int(state.epoch)} COMPLETE | Time: {elapsed:.1f}s | GPU mem: {mem_gb:.2f}GB")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            self.step_logs.append({
                "step": state.global_step,
                "epoch": state.epoch,
                **logs
            })
            if state.global_step % 50 == 0:
                loss = logs.get("loss", "n/a")
                lr   = logs.get("learning_rate", "n/a")
                logger.info(f"  Step {state.global_step:4d} | loss={loss} | lr={lr}")

# CELL 10 — REWARD FUNCTION TEST (run before training)

In [28]:
def test_reward_functions():
    """
    Sanity-check all three reward functions before training.
    Run this cell first to confirm rewards work correctly.
    """
    print("\n" + "="*60)
    print("REWARD FUNCTION TESTS")
    print("="*60)

    test_cases = [
        ("hello world",          "hello world"),        # perfect
        ("hello word",           "hello world"),        # one error
        ("the plaintiff filed",  "the plaintiff filed"), # domain term correct
        ("the plantiff filed",   "the plaintiff filed"), # domain term wrong
        ("completely wrong text","hello world"),         # very bad
    ]

    print("\n--- MWER Rewards ---")
    hyps = [t[0] for t in test_cases]
    refs = [t[1] for t in test_cases]
    mwer_r = compute_mwer_reward(hyps, refs)
    for (h, r), reward in zip(test_cases, mwer_r):
        print(f"  hyp: '{h}' | ref: '{r}' => reward={reward:.3f}")

    print("\n--- WWER Rewards (domain terms weighted 3x) ---")
    wwer_r = compute_wwer_reward(hyps, refs)
    for (h, r), reward in zip(test_cases, wwer_r):
        print(f"  hyp: '{h}' | ref: '{r}' => reward={reward:.3f}")

    print("\n--- LLM Rewards (mock) ---")
    llm_r = compute_llm_reward(hyps, refs, use_mock=True)
    for (h, r), reward in zip(test_cases, llm_r):
        print(f"  hyp: '{h}' | ref: '{r}' => reward={reward:.3f}")

    print("\n[PASS] All reward functions returned tensors of correct shape")
    print("Note: WWER should penalise 'plantiff' harder than MWER — check above!")
    return mwer_r, wwer_r, llm_r

# CELL 11 — MAIN TRAINING PIPELINE


In [29]:
from transformers.utils.notebook import NotebookProgressCallback

In [30]:
def run_baseline_training():
    """
    Full pipeline:
      1. Load data
      2. Preprocess
      3. Load model
      4. Run SFT baseline (CE loss only)
      5. Log hook points
      6. Evaluate WER

    After this works: activate reward injection in compute_loss() above.
    """

    # 1. Load data
    # dataset = load_common_voice(TRAIN_SAMPLES, EVAL_SAMPLES)
    dataset = load_dataset_librispeech(TRAIN_SAMPLES, EVAL_SAMPLES)

    # 2. Build processor
    processor = build_processor()

    # 3. Preprocess
    logger.info("Preprocessing dataset ...")
    # dataset = dataset.map(
    #     lambda b: prepare_dataset(b, processor),
    #     remove_columns=dataset["train"].column_names,
    #     num_proc=2,
    # )

    dataset = dataset.map(
        lambda b: prepare_dataset(b, processor),
        remove_columns=dataset["train"].column_names,
        num_proc=1,   # also drop this to 1 — multiprocessing can cause issues in Colab
    )

    # 4. Load model
    logger.info(f"Loading model: {SFT_CHECKPOINT}")
    # model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    model = WhisperForConditionalGeneration.from_pretrained(SFT_CHECKPOINT)

    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens    = []
    model.generation_config.language           = LANGUAGE
    model.generation_config.task               = TASK
    model.generation_config.max_length         = 448

    # 5. Data collator
    collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

    # 6. Training arguments
    # training_args = Seq2SeqTrainingArguments(
    #     output_dir=OUTPUT_DIR,
    #     per_device_train_batch_size=BATCH_SIZE,
    #     per_device_eval_batch_size=BATCH_SIZE,
    #     gradient_accumulation_steps=1,
    #     learning_rate=LEARNING_RATE,
    #     warmup_steps=100,
    #     num_train_epochs=NUM_EPOCHS,
    #     gradient_checkpointing=True,
    #     fp16=True,
    #     eval_strategy="epoch",        # <-- was evaluation_strategy
    #     save_strategy="epoch",
    #     logging_steps=25,
    #     predict_with_generate=True,
    #     generation_max_length=225,
    #     load_best_model_at_end=True,
    #     metric_for_best_model="wer",
    #     greater_is_better=False,
    #     report_to=["tensorboard"],
    #     dataloader_num_workers=2,
    # )
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1,
        learning_rate=LEARNING_RATE,
        warmup_steps=10,           # FIX 1: was 100, must be << total_steps (65)
        num_train_epochs=NUM_EPOCHS,
        gradient_checkpointing=True,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,          # also fix this — was 25, too infrequent for 13 steps/epoch
        predict_with_generate=True,
        generation_max_length=100, # FIX 3: match max_new_tokens in compute_loss
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        report_to=["tensorboard"],
        dataloader_num_workers=2,
    )

    # 7. Build trainer with hook-point logger
    logger.info("Initialising RewardAwareTrainer ...")

    # trainer = RewardAwareTrainer(
    #     model=model,
    #     args=training_args,
    #     train_dataset=dataset["train"],
    #     eval_dataset=dataset["validation"],
    #     processing_class=processor.feature_extractor,  # <-- was tokenizer=
    #     data_collator=collator,
    #     compute_metrics=lambda pred: compute_metrics(pred, processor),
    #     callbacks=[TrainingLogger()],
    #     processor=processor,
    #     reward_mode=REWARD_MODE,
    # )

    trainer = RewardAwareTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        processing_class=processor.feature_extractor,
        data_collator=collator,
        compute_metrics=lambda pred: compute_metrics(pred, processor),
        callbacks=[TrainingLogger()],
        processor=processor,
        reward_mode=REWARD_MODE,
        reward_weight=0.05,        # FIX 2: was defaulting to 0.3 — must pass explicitly
    )

    # 8. Train
    logger.info("Starting training ...")
    train_result = trainer.train()

    # 9. Save & report
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)

    metrics = train_result.metrics
    logger.info(f"\nTraining complete!")
    logger.info(f"  Train loss : {metrics.get('train_loss', 'n/a'):.4f}")
    logger.info(f"  Train time : {metrics.get('train_runtime', 0):.1f}s")
    logger.info(f"  Samples/sec: {metrics.get('train_samples_per_second', 0):.1f}")

    # 10. Final eval

    # eval_results = trainer.evaluate()
    # logger.info(f"  Final WER  : {eval_results.get('eval_wer', 'n/a'):.2f}%")

    # return trainer, eval_results

    logger.info("Running final evaluation on best checkpoint ...")
    best_ckpt = trainer.state.best_model_checkpoint
    if best_ckpt:
        logger.info(f"  Loading best checkpoint: {best_ckpt}")
        best_model = WhisperForConditionalGeneration.from_pretrained(best_ckpt)
        best_model.generation_config.forced_decoder_ids = None
        best_model.generation_config.suppress_tokens    = []
        best_model.generation_config.language           = LANGUAGE
        best_model.generation_config.task               = TASK


        eval_trainer = Seq2SeqTrainer(
            model=best_model,
            args=training_args,
            eval_dataset=dataset["validation"],
            processing_class=processor.feature_extractor,
            data_collator=collator,
            compute_metrics=lambda pred: compute_metrics(pred, processor),
        )
        eval_trainer.remove_callback(NotebookProgressCallback)
        eval_results = eval_trainer.evaluate()

        # eval_trainer = Seq2SeqTrainer(
        #     model=best_model,
        #     args=training_args,
        #     eval_dataset=dataset["validation"],
        #     processing_class=processor.feature_extractor,
        #     data_collator=collator,
        #     compute_metrics=lambda pred: compute_metrics(pred, processor),
        # )
        # eval_results = eval_trainer.evaluate()
    else:
        # No checkpoint saved (e.g. only 1 epoch) — evaluate current model
        logger.info("  No best checkpoint found, evaluating current model state ...")
        # Reinitialise a clean trainer with current model to avoid state corruption
        # eval_trainer = Seq2SeqTrainer(
        #     model=model,
        #     args=training_args,
        #     eval_dataset=dataset["validation"],
        #     processing_class=processor.feature_extractor,
        #     data_collator=collator,
        #     compute_metrics=lambda pred: compute_metrics(pred, processor),
        # )
        # eval_results = eval_trainer.evaluate()
        eval_trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            eval_dataset=dataset["validation"],
            processing_class=processor.feature_extractor,
            data_collator=collator,
            compute_metrics=lambda pred: compute_metrics(pred, processor),
        )
        eval_trainer.remove_callback(NotebookProgressCallback)
        eval_results = eval_trainer.evaluate()

    logger.info(f"  Final WER  : {eval_results.get('eval_wer', 'n/a'):.2f}%")
    return trainer, eval_results

# CELL 12 — RUN EVERYTHING


# Running only SFT - latest

In [48]:
if __name__ == "__main__":

    print("\n" + "="*60)
    print("STEP 1: Testing reward functions ...")
    print("="*60)
    test_reward_functions()

    print("\n" + "="*60)
    print("STEP 2: Running baseline training ...")
    print("="*60)
    trainer, eval_results = run_baseline_training()

    print("\n" + "="*60)
    print("DONE. Next step: uncomment reward injection in compute_loss()")
    print("="*60)


STEP 1: Testing reward functions ...

REWARD FUNCTION TESTS

--- MWER Rewards ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.667
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- WWER Rewards (domain terms weighted 3x) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.400
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- LLM Rewards (mock) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.493
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => rewa

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Wer
1,No log,1.573622,99.897300
2,1.107011,1.313696,99.845900
3,1.107011,1.073249,99.743200
4,0.699511,0.733013,99.794600
5,0.699511,0.552767,51.232700


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]


DONE. Next step: uncomment reward injection in compute_loss()


# OG running only SFT (old)

In [39]:
if __name__ == "__main__":

    print("\n" + "="*60)
    print("STEP 1: Testing reward functions ...")
    print("="*60)
    test_reward_functions()

    print("\n" + "="*60)
    print("STEP 2: Running baseline training ...")
    print("="*60)
    trainer, eval_results = run_baseline_training()

    print("\n" + "="*60)
    print("DONE. Next step: uncomment reward injection in compute_loss()")
    print("="*60)


STEP 1: Testing reward functions ...

REWARD FUNCTION TESTS

--- MWER Rewards ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.667
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- WWER Rewards (domain terms weighted 3x) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.400
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- LLM Rewards (mock) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.493
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => rewa

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Wer
1,No log,1.573554,99.871600
2,1.107018,1.313742,99.845900
3,1.107018,1.073212,99.743200
4,0.699490,0.732953,99.794600
5,0.699490,0.552710,51.361100


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]


DONE. Next step: uncomment reward injection in compute_loss()


# RL with SFT

In [42]:
if __name__ == "__main__":

    print("\n" + "="*60)
    print("UNCOMMENTED reward injection in compute_loss() on OG model without SFT")
    print("="*60)

    print("\n" + "="*60)
    print("STEP 1: Testing reward functions ...")
    print("="*60)
    test_reward_functions()

    print("\n" + "="*60)
    print("STEP 2: Running baseline training ...")
    print("="*60)
    trainer, eval_results = run_baseline_training()


UNCOMMENTED reward injection in compute_loss() on OG model without SFT

STEP 1: Testing reward functions ...

REWARD FUNCTION TESTS

--- MWER Rewards ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.667
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- WWER Rewards (domain terms weighted 3x) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.400
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- LLM Rewards (mock) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Wer
1,No log,1.573551,99.871600
2,1.106977,1.313802,99.845900
3,1.106977,1.073239,99.768900
4,0.699501,0.732871,99.794600
5,0.699501,0.552547,51.181300


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]